# Welcome to Week 4 - LangChain and LangGraph

## Lab 1: Abstraction Levels and the building blocks

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The LangChain docs</h2>
            <span style="color:#00bfff;">The documentation lives at <a href="https://docs.langchain.com/oss/python/">https://docs.langchain.com/oss/python/</a>. It is worth a browse. Watch out for older 0.x material that still floats around the web, because the modern API is quite different.
            </span>
        </td>
    </tr>
</table>

## The four levels of abstraction

LangChain and LangGraph form 4 levels of abstraction, with each one built on the ones before. The terminology is confusing because 'LangChain' appears in a few places..

| Layer | Packages | What it gives you | What you control |
|---|---|---|---|
| 1. Building blocks | `langchain-core` + `langchain-openai` | chat models, the `@tool` decorator, messages, structured output | everything, including the tool loop by hand |
| 2. Orchestration | `langgraph` | a graph of steps, with state, memory and checkpointing | the control flow (you design the graph) |
| 3. Agent | `langchain` (`create_agent`) | the standard agent loop, prebuilt | just model, tools and a prompt |
| 4. Harness | `deepagents` (`create_deep_agent`) | an opinionated harness with planning, sub-agents and a filesystem | your intent |


## Order of play for this week:

DAY 1 (today): The building blocks  
DAY 2: LangGraph  
DAY 3: LangChain create_agent  
DAY 4: Deep Agents  
DAY 5: The Sidekick project  

## Today is Layer 1: the building blocks

This is where LangChain began: an abstraction layer, not unlike LiteLLM - but a more heavyweight version.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

### A first model call

`ChatOpenAI` is the abstraction around OpenAI calls, and there are similar packages for other LLM providers.

We call `invoke` with a prompt; this is a key method in LangChain.

In [2]:
llm = ChatOpenAI(model="gpt-5.4-mini")

message = "What does it mean for an AI Agent to be autonomous"

reply = llm.invoke(message)

print(reply.content)

An AI agent is **autonomous** if it can **act on its own to achieve goals**, with **minimal human intervention**.

In practice, that means it can:

- **Perceive** its environment or inputs
- **Decide** what to do next
- **Take actions** without waiting for step-by-step instructions
- **Adapt** when conditions change
- **Work toward a goal** over multiple steps

### Example
A non-autonomous AI might answer only when asked a question.  
An autonomous AI agent might:
1. notice a task,
2. plan a sequence of actions,
3. use tools or APIs,
4. correct mistakes,
5. keep going until the goal is completed.

### Important nuance
Autonomy is usually **partial**, not absolute. Most AI agents are autonomous only within **constraints** set by humans, such as:
- allowed tools
- time limits
- safety rules
- budget
- approval steps for risky actions

So, autonomy means the agent has some degree of **independent decision-making and action**, not that it is fully self-governing like a human.


### Streaming

For a live, token by token feel, swap `invoke` for `stream` and loop over the chunks

In [3]:
for chunk in llm.stream("What does it mean for an AI Agent to be autonomous"):
    print(chunk.content, end="", flush=True)

An AI agent is **autonomous** if it can **perceive its environment, decide what to do, and act toward a goal with little or no step-by-step human control**.

In practice, autonomy usually means the agent can:
- **Observe** what’s happening
- **Choose actions** based on its goals and current situation
- **Carry out those actions**
- **Adjust** when conditions change
- **Work continuously** without needing constant instructions

### Example
A chatbot that only answers when asked is not very autonomous.  
A robot vacuum that maps a room, decides where to clean next, avoids obstacles, and returns to charge by itself is more autonomous.

### Important nuance
Autonomy does **not** mean the agent is independent of humans in every sense. It may still:
- be given goals by humans
- operate within rules or limits
- require oversight, updates, or intervention

So, autonomy is about **self-directed action within a defined scope**, not complete freedom.

### Any OpenAI-compatible provider

As before; we can use OpenAI compatible endpoints with the ChatOpenAI object

In [ ]:
openrouter_llm = ChatOpenAI(
    model="claude-opus-4-8",
    base_url="https://api.anthropic.com/v1/",
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

reply = openrouter_llm.invoke("What is Deep Agents?")
print(reply.content)

"Deep Agents" usually refers to one of two closely related things:

## 1. The concept

A **deep agent** is an AI agent designed for long-horizon, complex tasks — as opposed to a "shallow" agent, which is just an LLM in a loop calling tools. Shallow agents work fine for short tasks but fall apart on work that takes hundreds of steps, spans many files, or requires sustained strategy.

The term was popularized by LangChain (around mid-2025), reverse-engineering what made systems like **Claude Code**, **Deep Research** tools, and **Manus** work well. They identified four recurring architectural ingredients:

- **A detailed system prompt** — long, instructive, often with examples and heuristics, rather than a one-liner.
- **Planning tools** — e.g. a `write_todos` tool that lets the agent maintain and revise a task list. Even though writing a todo list does nothing externally, it keeps the agent's goals salient in context and helps it recover after long tangents.
- **Sub-agents** — spawning 

### Messages

LangChain comes with abstractions around SystemMessage, HumanMessage, AIMessage, although you can use the usual list-of-dicts instead.

In [ ]:
messages = [
    SystemMessage("You are a poet. You answer questions in Dactylic hexameter, with 10 syllables per line."),
    HumanMessage("Tell me about the future of AI agents, an epic poem in 100 lines."),
]

print(llm.invoke(messages).content)

# The exact same call using plain dictionaries, the format you already know
messages_as_dicts = [
    {"role": "system", "content": "You are a poet. You answer questions in Dactylic hexameter, with 10 syllables per line."},
    {"role": "user", "content": "Tell me about films in the 1910s"},
]
print(llm.invoke(messages_as_dicts).content)

### Tools with the `@tool` decorator

A tool is a Python function the model is allowed to call. The modern way to make one is the `@tool` decorator. Your docstring becomes the description the model reads, and your type hints become the argument schema, just like `@function_tool` with OpenAI Agents SDK.

This replaces the older `Tool(...)` wrapper from earlier versions of LangChain.

In [ ]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print("name:", get_share_price.name)
print("description:", get_share_price.description)
print("args:", get_share_price.args)
print("called directly:", get_share_price.invoke({"symbol": "AAPL"}))

### Giving tools to the model

This is a bit clunky; redemption will come in Day 3 when we use the create_agent features,

For now, we have to write the loop ourselves, rather like we did in Week 1.

The first step is to bind the tools to the model with `bind_tools`. Now when we invoke, the model may come back not with an answer but with a request to run a tool. That request shows up in `.tool_calls`.

In [ ]:
llm_with_tools = llm.bind_tools([get_share_price])

response = llm_with_tools.invoke("What is the share price of Amazon?")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)

### Running the tool loop by hand

So now we need to write a little loop, quite similar to Week 1

In [ ]:
# Start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("What is the share price of Amazon?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# Run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

# Invoke again, now that the model can see the tool result
final = llm_with_tools.invoke(conversation)
print(final.content)

### Structured output

Similar to OpenAI Agents SDK, we can require the model to respond with a Pydantic subclass

In [ ]:
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

structured_llm = llm.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Amazon the technology company")
print(company)
print("Just the ticker:", company.ticker)

## That is Layer 1

It's like a more rich and more involved version of LiteLLM.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Write a second tool of your own, perhaps one that looks up a fictional weather report for a city. Bind both tools to the model and ask a question that needs both, then run the tool loop by hand until the model gives a final answer. It will feel more gritty than OpenAI Agents SDK or CrewAI.
            We will fix that by Day 3!
            </span>
        </td>
    </tr>
</table>